# Week 2 Seminar Notebook
This week we are going to be thinking about and working with n-gram language models, practising some Python skills along the way.

We start by building a **character-based language model** from a sample text. Then you will play the **Shannon game** (to be explained), and finally we will use both to estimate the **entropy of English** and compare the results.

## A character-based language model

Our sample text is the first chapter of *Crime and Punishment*. We will count how often each character, and each pair of characters, occurs in it, and use these counts to assign probabilities to characters and to whole sentences.

In [ ]:
# download the book from Project Gutenberg (the ! runs a command outside Python)
!wget https://www.gutenberg.org/files/2554/2554-0.txt
# read the whole file into one string
with open('2554-0.txt') as fp:
    c_and_p = fp.read()
# find where chapter one starts and ends: from just after the heading 'CHAPTER I'
# up to the heading 'CHAPTER II' (Activity 2 looks at this more closely)
start = c_and_p.find('CHAPTER I\n') + len('CHAPTER I\n')
end = c_and_p.find('CHAPTER II\n')
# select chapter one and convert it to lower case
raw = c_and_p[start:end].lower()
# keep only the letters a-z and the space, using a for loop
allowed = 'abcdefghijklmnopqrstuvwxyz '
kept = []
for ch in raw:
    if ch in allowed:
        kept.append(ch)
chapter_one = ''.join(kept)       # glue the kept characters back into one string

### Python practice: files and strings

Before we start counting, let's practise some Python from the Week 2 Python workbook (LELA60341) on the text we just downloaded. Each **Activity** is short: replace every `???` with your own code and run the cell. Parts marked ⭐ **Challenge** are optional.

**Activity 1 (reading a file line by line)**

Above, we read the whole file into one long string with `fp.read()`. `fp.readlines()` instead gives a **list** of lines.

Fill in the blank to print how many lines the file has, then run the cell to see lines 100 to 109.

In [ ]:
with open('2554-0.txt') as fp:
    lines = fp.readlines()

print('Number of lines:', ???)    # hint: len() gives the length of a list

for line in lines[100:110]:
    print(line.strip())

**Activity 2 (checking where chapter one starts and ends)**

In the first code cell, `.find()` gave us the positions `start` and `end`. Let's check them by looking at the text either side.

The first `print` shows the 60 characters just **after** `start`. Fill in the slice for the 60 characters just **before** `end`.

In [ ]:
print(start, end)
print(c_and_p[start:start+60])    # the 60 characters after start
print('-----')
print(c_and_p[???:???])           # the 60 characters before end

`.find()` only gives the **first** match. The cell below uses a `while` loop to find **every** `'CHAPTER II\n'` in the book. Run it, then think about these questions:

* Why are there several matches? Why does our code in the first code cell still pick the right one?
* Why do we search for `'CHAPTER II\n'` (with a newline) and not just `'CHAPTER II'`? (What else starts with those letters?)

In [ ]:
positions = []
pos = c_and_p.find('CHAPTER II\n')
while pos != -1:                                   # -1 means "not found"
    positions.append(pos)
    pos = c_and_p.find('CHAPTER II\n', pos + 1)    # search again, just after this match

for p in positions:
    print(p, c_and_p[p:p+40])

**Activity 3 (improving the cleaning)**

The first line of the cell below prints the start of `chapter_one`. The newline character `'\n'` is not in `allowed`, so it was thrown away, and the last word of each line got stuck to the first word of the next.

Fill in the blank so that every newline is replaced by a space **before** the loop. Then compare the number of words before and after. Why are there more words afterwards?

In [ ]:
print(chapter_one[:300])

raw2 = c_and_p[start:end].lower()
raw2 = raw2.replace(???, ' ')     # turn every newline into a space

kept = []
for ch in raw2:
    if ch in allowed:
        kept.append(ch)
clean = ''.join(kept)

print(len(chapter_one.split()), 'words before')
print(len(clean.split()), 'words after')

**Activity 4 (counting with a string method)**

The first line counts how many times `'the'` occurs. Fill in the second line to count `' the '`, with a space either side. Why are the two numbers different?

In [ ]:
print(chapter_one.count('the'))
print(chapter_one.count(???))

We are going to be assigning probabilities to sentences using bigram models, so we need to extract unigram and bigram counts. We will store them in dictionaries.

In [ ]:
# Count how many times each character occurs, using the dictionary counting pattern from the Week 2 Python workbook
unigrams = {}
for ch in chapter_one:
    unigrams[ch] = unigrams.get(ch, 0) + 1

# the total number of characters
total_unigrams = len(chapter_one)

In [ ]:
unigrams

**Activity 5 (counting with if/else)**

The counting cell above counts characters with `unigrams.get(ch, 0) + 1`. Here is the same idea written with `if`/`else`: if we have seen the character before, add one to its count; otherwise this is the first time, so its count is 1.

Fill in the two blanks. The cell should print `True`.

In [ ]:
my_counts = {}
for ch in chapter_one:
    if ch in my_counts:
        my_counts[ch] = ???     # add one to the count we already have
    else:
        my_counts[ch] = ???     # first time we have seen this character

print(my_counts == unigrams)

**Activity 6 (probabilities of characters)**

The probability of a character is its count divided by the total number of characters. The first two lines work out P(e) as an example.

1. Work out P(t) and P(space) in the same way. (The space character is written `' '`.)
2. The probability of a vowel is P(a) + P(e) + P(i) + P(o) + P(u). Fill in the blank in the loop.
3. The probability of **not** getting a space is 1 − P(space).

In [ ]:
p_e = unigrams['e'] / total_unigrams
print('P(e) =', round(p_e, 3))

# 1.
p_t = ???
p_space = ???
print('P(t) =', round(p_t, 3), '  P(space) =', round(p_space, 3))

# 2.
p_vowel = 0.0
for v in 'aeiou':
    p_vowel += ???          # add the probability of this vowel
print('P(vowel) =', round(p_vowel, 3))

# 3.
print('P(not space) =', round(???, 3))

⭐ **Challenge (optional):** print the 10 most frequent characters. We build a list of `(count, character)` pairs and sort it from largest to smallest. Fill in the two blanks.

In [ ]:
pairs = []
for char, count in unigrams.items():
    pairs.append((count, char))
pairs = sorted(pairs, reverse=True)       # tuples sort by their first element: the count

for count, char in pairs[???]:            # the first 10 pairs
    print(f"'{char}'", count, round(???, 3))   # the probability of this character

In [ ]:
# Count each pair of neighbouring characters (a bigram).
# We need the position i, so we use an index-based loop.
# There is one bigram fewer than there are characters.
bigrams = {}
total_bigrams = len(chapter_one) - 1
for i in range(total_bigrams):
    pair = chapter_one[i:i+2]        # the character at position i and the one after it
    bigrams[pair] = bigrams.get(pair, 0) + 1

**Activity 7 (conditional probability)**

The bigram counts tell us how likely a character is **given** the one before it:

$$P(h \mid t) = \frac{\text{count of } th}{\text{count of } t}$$

The first two lines work this out as an example.

1. Work out P(h | s) in the same way.
2. Work out P(h), the plain probability of `h` (as in Activity 6).
3. Compare the three numbers. Is `h` much more likely after `t`? What does this tell us about whether letters are independent of each other?

In [ ]:
p_h_given_t = bigrams['th'] / unigrams['t']
print('P(h | t) =', round(p_h_given_t, 3))

# 1.
p_h_given_s = ???
print('P(h | s) =', round(p_h_given_s, 3))

# 2.
p_h = ???
print('P(h)     =', round(p_h, 3))

Which characters can follow `q`? This loop looks at every bigram and prints those whose first character is `q`, with P(next | q). Fill in the blank and run it. What do you notice?

In [ ]:
for pair, count in bigrams.items():
    if pair[0] == 'q':
        print(f"'{pair[1]}'", round(count / unigrams[???], 3))

**Activity 8 (saving counts to a file)**

Counting takes time on large texts, so we often save the counts to a file.

1. Fill in the blank to write one line per character in the form `character<TAB>count`. `fp.write()` only accepts strings, so the count has to be converted with `str()`. `'\t'` is a tab and `'\n'` a newline.
2. The second part reads the file back and shows the first 5 lines. How many lines should the file have, and why?

In [ ]:
# 1. write
with open('char_unigrams.tsv', 'w') as fp:
    for char, count in unigrams.items():
        fp.write(char + '\t' + ??? + '\n')

# 2. read back
with open('char_unigrams.tsv') as fp:
    lines = fp.readlines()
print(len(lines), 'lines')
for line in lines[:5]:
    print(line.rstrip())

⭐ **Challenge (optional):** rebuild the counts dictionary from `lines`. `split('\t')` separates the character from the count, and `int()` turns the count back into a number.

In [ ]:
loaded = {}
for line in lines:
    fields = line.rstrip().split('\t')     # e.g. ['e', '2190']
    loaded[fields[0]] = ???

print(loaded == unigrams)

We can now use the counts to calculate the probability of a sentence, including sentences that weren't in the text the model was built from. We add up the negative log probabilities (the *surprisal*) of each character given the one before it:

In [ ]:
import math
# math.log(x, 2) is the logarithm of x to base 2
# The sentence is a string, so each element is a character.
# It starts with a space, which marks the start of the first word.
sentence = " the cat sat on the mat"
log_prob = 0.0
prob = 1.0
for i in range(len(sentence) - 1):
    pair = sentence[i:i+2]
    p = bigrams.get(pair, 0) / unigrams[pair[0]]      # P(next character | this character)
    log_prob += -math.log(p, 2)
    prob *= p
log_prob

## Perplexity

Raw log probability of a text grows more negative the longer the text is, so it can't fairly compare texts or models. **Perplexity** fixes this by averaging per token and putting the result on an intuitive scale.

### Cross entropy

The average negative log probability the model gave the true tokens:

$$
H = -\frac{1}{N} \sum_{i=1}^{N} \log p(x_i \mid x_{<i})
$$

### Perplexity

The exponential of cross entropy, using the same log base:

$$
\text{PPL} = e^{H}
$$

It's the **effective number of equally likely choices** the model was torn between at each step:

- **PPL = 1:** the model was certain and right every time.
- **PPL = 20:** the model was as uncertain as picking uniformly among 20 options.

> **Note:** Values are only comparable across models that share a tokenizer and are evaluated on the same text.

**Activity 9 (perplexity)**

`log_prob` from the sentence cell above is the **total** surprisal of the sentence. To compare sentences of different lengths we use the **average** surprisal per bigram (the cross-entropy), and then "perplexity":

* cross-entropy = `log_prob` ÷ number of bigrams
* perplexity = 2 to the power of the cross-entropy (`2 ** x` in Python)

1. Fill in the two blanks and run the cell.
2. Now change `sentence` in the cell above to `" sat the on mat cat the"` (the same words in a jumbled order), run that cell again, then run this one again. Which sentence is more surprising to the model? Why?

In [ ]:
n_bigrams = len(sentence) - 1
cross_entropy = ???      # log_prob divided by n_bigrams
perplexity = ???         # 2 to the power of cross_entropy
print(sentence, '|', round(cross_entropy, 2), '|', round(perplexity, 2))

## The Shannon game

Shannon (1951) described an ingenious way of estimating the entropy of English by using human predictions. A person guesses a text one character at a time, and the number of guesses they need for each character tells us how predictable English is.

Shannon, C. E. (1951). Prediction and entropy of printed English. Bell system technical journal, 30(1), 50-64.

**Play it:** run the cell below (click ▶). It shows the text so far and asks you to guess the next character: a letter, or the word `space`. Keep guessing until you get it right, and the game moves on to the next character. At the end it shows how many guesses each character took and turns this into an estimate of the entropy of English, in bits per character.

Compare your estimate with your neighbours'. Keep a note of it: next we will estimate the entropy of English from our character model, and compare the two.

In [ ]:
#@title ▶ Run this cell to play the Shannon game { display-mode: "form" }
# (Double-click here if you want to see how the game works.)
import math

sg_text = "the old dog slept by the fire all afternoon"
sg_alphabet = "abcdefghijklmnopqrstuvwxyz "

print("Guess the text one character at a time.")
print("Type a letter (a-z), or type 'space' for a space, then press Enter.")
print("Type 'quit' to stop early.\n")

sg_guesses = []          # how many guesses each character took
sg_quit = False
for sg_pos in range(len(sg_text)):
    sg_target = sg_text[sg_pos]
    sg_tried = []
    while True:
        sg_reply = input(sg_text[:sg_pos] + "_    guess " + str(len(sg_tried) + 1) + ": ").lower()
        if sg_reply == "quit":
            sg_quit = True
            break
        if sg_reply == "space":
            sg_reply = " "
        if len(sg_reply) != 1 or sg_reply not in sg_alphabet:
            print("   Please type a single letter, or the word 'space'.")
            continue
        if sg_reply in sg_tried:
            print("   You have already tried that one.")
            continue
        sg_tried.append(sg_reply)
        if sg_reply == sg_target:
            print("   ✓ correct!")
            break
        else:
            print("   ✗ no")
    if sg_quit:
        break
    sg_guesses.append(len(sg_tried))

print()
print("Text so far:", sg_text[:len(sg_guesses)])
if len(sg_guesses) > 0:
    # how often did the right answer come on the 1st guess, the 2nd guess, ...?
    sg_counts = {}
    for g in sg_guesses:
        sg_counts[g] = sg_counts.get(g, 0) + 1
    print("Number of guesses needed : how many characters")
    for g in sorted(sg_counts):
        print(f"   {g:2} guess(es) : {sg_counts[g]}")

    # Shannon's estimate: the entropy of the number-of-guesses distribution
    n = len(sg_guesses)
    shannon_upper = 0.0
    for g in sg_counts:
        q = sg_counts[g] / n
        shannon_upper += -q * math.log(q, 2)

    print()
    print("Average number of guesses per character:", round(sum(sg_guesses) / n, 2))
    print("Your estimate of the entropy of English: at most", round(shannon_upper, 2), "bits per character")
    print("(This is Shannon's upper bound. The better your guesses, the lower it gets.)")

**Optional: the full version at home.** The original Shannon game program lets you play with longer texts. To play it in your own time:

Copy this URL into a browser window and press return

https://github.com/lianghuang3/shannon_game/archive/refs/heads/main.zip

Unpack repository to somewhere on local machine

If you are on University Computer or have VS Code installed
Open VSCode (e.g. via Anaconda Navigator)
Select File -> Open Folder and then select the shannon_game folder

Open shannon_game.py and press Run

Follow instructions given by the program once running.

## Calculate Entropy of English from a sample text

How do the estimates from our character model compare with your Shannon game estimate?

Now we want to use these counts to calculate the entropy of English.

If we were to calculate the entropy assuming every character was independent and occurred an equal number of times, the calculation would be very simple:

In [ ]:
import math
# math.log(x, 2) is the logarithm of x to base 2
-math.log(1/27, 2)

We know that letters do not occur the same number of times (are not equiprobable) so the next thing we might try is to calculate the entropy reflecting the unequal rates but still assuming independence.

In [ ]:
import math
# Entropy: add up -p * log2(p) over every character. math.log(p, 2) is log to base 2.
H = 0.0
for ch, count in unigrams.items():
    p = count / total_unigrams       # how often this character occurs
    H += -p * math.log(p, 2)
H

We know that characters are not independent - for example, h is more likely to occur following t than following x. This is why we use bigram probabilities.

In [ ]:
H = 0.0
for pair, count in bigrams.items():
    p_pair = count / total_bigrams          # P(x, y): how often this pair occurs
    p_next = count / unigrams[pair[0]]      # P(y | x): how often y follows x
    H += -p_pair * math.log(p_next, 2)
H

How does this estimate compare with your Shannon game estimate? Shannon's own experiments suggested that English carries roughly 1 bit per character or less. Why might our bigram model's estimate be so much higher?

**Activity 10 (trigram entropy)**

Estimate entropy from a trigram character model. First of all you will need to rewrite the code to extract trigram counts.

In [ ]:
# Count each sequence of three neighbouring characters (a trigram)
trigrams = {}
total_trigrams = ???
for i in range(total_trigrams):
    triple = chapter_one[???]
    trigrams[triple] = trigrams.get(triple, 0) + 1

Next you will need to use these to calculate the trigram probabilities and combine to calculate the entropy.

In [ ]:
H = 0.0
for triple, count in trigrams.items():
    p_triple = count / ???                  # P(x, y, z)
    p_next = count / bigrams[???]           # P(z | x, y)
    H += -p_triple * math.log(p_next, 2)
H

## Language modelling for words

We are now going to switch to building and using word-based language models. Instead of a string of characters we need a **list of words**. We tidy the text with the string methods from the Week 2 Python workbook (`lower()`, `replace()`) and then use `split()`, which breaks a string at whitespace into a list of words and ignores repeated spaces.

In [ ]:
# c_and_p (the whole book), start, end and allowed were set in the first code cell
raw = c_and_p[start:end].lower()
# line breaks become spaces
raw = raw.replace('\n', ' ')
# mark the end of each sentence with the special token 'eol'
raw = raw.replace('. ', ' eol ')
# keep only a-z and space, as before
kept = []
for ch in raw:
    if ch in allowed:
        kept.append(ch)
# glue the characters back together, then split at whitespace to get a list of words
chapter_one = ''.join(kept).split()

With the text in this new format, the same counting code works. The only difference is that each element is now a word rather than a character.

In [ ]:
unigrams = {}
for word in chapter_one:
    unigrams[word] = unigrams.get(word, 0) + 1

total_unigrams = len(chapter_one)

In [ ]:
# A word bigram is two neighbouring words. We join them with a space
# so that the pair can be used as one dictionary key, e.g. 'the man'
bigrams = {}
total_bigrams = len(chapter_one) - 1
for i in range(total_bigrams):
    pair = " ".join(chapter_one[i:i+2])
    bigrams[pair] = bigrams.get(pair, 0) + 1

In [ ]:
unigrams

In [ ]:
bigrams

**Activity 11 (counting words)**

`chapter_one` is now a **list** of words.

1. `len()` gives the number of word *tokens*. Use `set()` to get the number of *different* words (types).
2. We replaced each full stop with the word `eol`. Use the list method `.count()` to count the sentences.
3. Fill in the blank to work out the average length of a word in characters. The loop skips the `eol` tokens with `continue`.

In [ ]:
# 1.
n_tokens = len(chapter_one)
n_types = ???            # hint: len() of a set of the words
print(n_tokens, 'tokens,', n_types, 'types')

# 2.
print('Sentences:', chapter_one.count(???))

# 3.
total_length = 0
n_words = 0
for w in chapter_one:
    if w == 'eol':
        continue                 # skip to the next word
    total_length += ???          # the number of characters in w
    n_words += 1
print('Average word length:', round(total_length / n_words, 2))

**Activity 12 (word probabilities)**

Word probabilities work just like character probabilities. P(the) is given as an example.

1. Work out P(old), the probability of the word `old`.
2. Work out P(old | the): the number of times `the old` occurs divided by the number of times `the` occurs. Remember that bigram keys are written with a space, like `'the old'`.
3. Fill in the blank in the loop, which prints every word that follows `the` at least 5 times, with P(word | the). Which words are most likely after `the`?

In [ ]:
p_the = unigrams['the'] / total_unigrams
print('P(the) =', round(p_the, 4))

# 1.
p_old = ???
print('P(old) =', round(p_old, 4))

# 2.
p_old_given_the = ???
print('P(old | the) =', round(p_old_given_the, 4))

# 3.
for pair, count in bigrams.items():
    first, second = pair.split(' ')
    if first == 'the' and count >= 5:
        print(second, round(count / ???, 3))

And the counts can be used to assign probabilities to sentences in exactly the same way:

In [ ]:
sentence = ["a", "man", "was", "in", "the", "house"]
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p = bigrams.get(pair, 0) / unigrams[sentence[i]]    # P(next word | this word)
    log_prob += -math.log(p, 2)
log_prob

**Activity 13 (the first word of a sentence)**

Because we are using bigrams the first element for which we calculate the probability for is the second word. However to give the probability for the full sentence we want to also consider the first word. Update the code so that it does this. You may want to make reference to the lecture slides for the week in order to remind yourself of how we do this.

In [ ]:
# 'eol' marks the end of a sentence, so it can also stand for the START of the next one.
# Add 'eol' to the beginning of the list, so that the first word also gets a probability.
# (We use a sentence starting with 'he': no sentence in chapter one starts with 'a'.)
sentence = ["he", "was", "in", "the", "house"]
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p = bigrams.get(pair, 0) / unigrams[sentence[i]]
    log_prob += -math.log(p, 2)
log_prob

Now try calculating the log probability for a different sentence of your own creation

In [ ]:
sentence = []    # put your own words in the list
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p = bigrams.get(pair, 0) / unigrams[sentence[i]]
    log_prob += -math.log(p, 2)
log_prob

This probably won't work because of unseen bigrams: a pair of words that never occurs in the chapter gets probability 0, and taking the log of 0 gives an error. (A word that never occurs at all gives a `KeyError`.)

We need to solve this problem via smoothing.

For example, add-one smoothing:

In [ ]:
sentence = ["a", "man", "was", "in", "the", "house", "when", "the", "day", "arrived"]
vocab_size = len(unigrams)          # the number of different words
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    # add one to every bigram count, and add the vocabulary size to the denominator
    p = (bigrams.get(pair, 0) + 1) / (unigrams.get(sentence[i], 0) + vocab_size)
    log_prob += -math.log(p, 2)
log_prob

Or alternatively, smoothing by interpolation, which mixes the bigram probability with the unigram probability of the next word:

In [ ]:
sentence = ["a", "man", "was", "in", "the", "house", "when", "the", "day", "arrived"]
# These lambdas can change but should sum to 1
lambda1 = 0.5
lambda2 = 1 - lambda1
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p_bigram = bigrams.get(pair, 0) / unigrams[sentence[i]]        # P(next word | this word)
    p_unigram = unigrams.get(sentence[i+1], 0) / total_unigrams    # P(next word)
    # combine the two, weighting each by its lambda
    p = lambda1 * p_bigram + lambda2 * p_unigram
    log_prob += -math.log(p, 2)
log_prob

### Putting it all together

**Activity 14 (scoring your own sentences)**

1. Replace each `???` with a sentence of your own (in lower case, without punctuation). The code writes them to a file, one per line.
2. The second part reads the file back and scores each sentence with the add-one smoothed model. Fill in the one blank: turn `line` into a list of words by making it lower case and then splitting it.
3. Run the cell. Which sentence does the model find least surprising (lowest perplexity)? Does a longer sentence always have a bigger total surprisal (`lp`)? Does it have a bigger perplexity?

In [ ]:
# 1. write your sentences to a file
my_sentences = ["the man was in the house",
                "???",
                "???",
                "???"]
with open('my_sentences.txt', 'w') as fp:
    for s in my_sentences:
        fp.write(s + '\n')

# 2. read them back and score each one
vocab_size = len(unigrams)
with open('my_sentences.txt') as fp:
    for line in fp:
        words = ???
        lp = 0.0
        for i in range(len(words) - 1):
            pair = " ".join(words[i:i+2])
            lp += -math.log((bigrams.get(pair, 0) + 1) / (unigrams.get(words[i], 0) + vocab_size), 2)
        perplexity = 2 ** (lp / (len(words) - 1))
        print(line.strip(), '|', round(lp, 2), '|', round(perplexity, 2))

⭐ **Challenge (optional) · Activity 15 (generating text)**

A language model can also *generate* text. Starting from the sentence-start token `eol`, this code repeatedly picks the **most likely** next word: the follower of the current word with the highest bigram count. This is called *greedy* generation.

Fill in the three blanks and run it. What happens? Why does greedy generation get stuck repeating itself? How might choosing the next word *at random*, in proportion to its probability, help?

In [ ]:
current = 'eol'
generated = []
for step in range(20):
    best_word = None
    best_count = 0
    for pair, count in bigrams.items():
        first, second = pair.split(' ')
        if first == current and count > best_count:   # a better follower than any so far
            best_word = second
            best_count = ???
    if best_word is None:      # dead end: no word ever follows this one
        break
    generated.append(???)
    current = ???              # move on to the word we just chose

print(' '.join(generated))